In [1]:
import torch
import myneural as mn
import torch.nn.functional as F
import time
import pyarrow.parquet as pq
import sentencepiece as spm
import math
torch.manual_seed(42)

In [2]:
# training tokenizer --------------
TRAIN = False
if TRAIN:
    pf = pq.ParquetFile('data/tinystories0.parquet')
    with open('data/input.txt', 'w', encoding='utf-8') as f:
        for i in range(100):
            text_list = pf.read_row_group(i).column('text').to_pylist() # 1000 rows each row group     
            for story in text_list:
                f.write(story)
                
    spm.SentencePieceTrainer.train( # pyright: ignore[reportAttributeAccessIssue]
        input = 'data/input.txt',
        model_prefix = 'models/token_model',
        vocab_size = 4096,
        model_type = 'bpe',
        character_coverage = 1.0,
        byte_fallback = True,
        add_dummy_prefix = True,
        )

In [3]:
sp = spm.SentencePieceProcessor(model_file="models/token_model.model") # pyright: ignore[reportCallIssue]
sp.Encode("Lily found a needle.") # pyright: ignore[reportAttributeAccessIssue]
print(''.join(sp.IdToPiece(id) for id in sp.Encode("Bob saw a shiny golden penny.\n He liked it.") + [sp.eos_id()] + sp.Encode("Lily found a flower.")).replace('▁', ' ').replace('</s>', '</s>\n')) # EOS IDs are hidden in general.

 Bob saw a shiny golden penny. He liked it.</s>
 Lily found a flower.


In [4]:
config = mn.Config()

In [5]:
config.n_embd = 256
config.n_head = 8
config.n_layer = 8
config.mini_batch_size = 128 # B
config.context_length = 256 # T
config.total_batch_size = 1 * config.mini_batch_size * config.context_length # MUST BE DIVISBLE by (B*T)
config.device = 'cuda'
config.vocab_size = 4096

max_iter = 6000
B = config.mini_batch_size
T = config.context_length
grad_accum_steps = config.total_batch_size // (B*T)
assert config.total_batch_size % (B * T) == 0, "Total batch size must be divisble by (B*T)"

In [6]:
import dataset
set = dataset.ParquetDataset([
    'data/tinystories0.parquet',
    ],
    B = config.mini_batch_size,
    T= config.context_length,
    tokenizer=sp,
    append_eos=True,
    loop_around=True,
    shuffle_rgs=True,
    seed=42,
)

dataloader = torch.utils.data.DataLoader(
    set,
    batch_size=None,# batch_size = None because i already give (B, T) in the dataset.
    # if i gave a batch_size, it would give me (batch_size, B, T)
    num_workers=6,
    pin_memory=True,
    prefetch_factor=4, # 2 batches will be loaded ahead of time.
    persistent_workers=True
) 

In [7]:
val_set = dataset.ParquetDataset([
    'data/tinystories0val.parquet',
    ],
    B = config.mini_batch_size,
    T= config.context_length,
    tokenizer=sp,
    append_eos=True,
    loop_around=False,
    shuffle_rgs=False,
    seed=42,
)

val_dataloader = torch.utils.data.DataLoader(
    val_set,
    batch_size=None,
    num_workers=0,
    pin_memory=True,
) 

In [8]:
def get_lr(step):
    max_lr = 1e-3
    min_lr = 1e-4
    warmup_steps=200
    max_steps=6000
    if step < warmup_steps:
        return max_lr * (step+1)/warmup_steps 
    if step > max_steps:
        return min_lr
    decay_ratio = (step-warmup_steps)/(max_steps-warmup_steps)
    coefficient = 0.5 * (1.0 + math.cos(decay_ratio * math.pi)) # starts at 1 reduces to 0
    return min_lr + coefficient * (max_lr - min_lr)
    

In [9]:
torch.set_float32_matmul_precision('high')
model = mn.Transformer(config, dtype=torch.float32, device=config.device)
params = model.parameters()

In [10]:
optimizer = model.configure_optimizer(weight_decay=0.1, learning_rate=get_lr(0), device=config.device)

In [11]:
# # Load the list of tensors
# loaded_params = torch.load("models/params_new.pth")

# # Manually re-assign them to your model
# for p, loaded_p in zip(model.parameters(), loaded_params):
#     p.data.copy_(loaded_p.data)

In [12]:
model_optim = torch.compile(model)
step = 0

In [13]:
def get_val_loss():
    max_batches = 25
    val_loss = []
    with torch.no_grad():
        for i, (xs, ys) in enumerate(val_dataloader):      
            if i >= max_batches:
                break
            xs = xs.to(device=config.device, non_blocking=True)
            ys= ys.to(device=config.device, non_blocking=True)
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                _, loss = model_optim(xs, targets=ys)
            val_loss.append(loss.item())
    return sum(val_loss)/len(val_loss)

In [15]:
data_iter = iter(dataloader)
lossi = []
timei = []
mean_stat_batches = 100
print_batches = 100
val_batches = 300
for _ in range(step, max_iter):
    t_start = time.time()
    optimizer.zero_grad(set_to_none=True)
    # forward pass
    loss_accum = torch.tensor(0.0, device=config.device)
    for micro_step in range(grad_accum_steps):
        xs, ys= next(data_iter) # loop_around=True so we don't need to worry about StopIteration exception
        xs = xs.to(device=config.device, non_blocking=True)
        ys= ys.to(device=config.device, non_blocking=True)
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            logits, loss = model_optim(xs, targets=ys)
        loss = loss / grad_accum_steps # normalizer to avoid loss gradients being exagerrated by grad_accum_steps times.
        loss_accum += loss.detach()
        # backward pass
        loss.backward()
    
    norm = torch.nn.utils.clip_grad_norm_(params, 1.0)
    
    # update
    lr = get_lr(step)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr
    optimizer.step()
    
    torch.cuda.synchronize()
    t_end = time.time()
    
    # track stats    
    lossi.append(loss_accum.item())
    timei.append(t_end-t_start)
    if step % print_batches == 0:
        lossi = lossi[-mean_stat_batches:]
        timei = timei[-mean_stat_batches:]
        train_loss = sum(lossi)/len(lossi)
        if step % val_batches == 0:
            val_loss = get_val_loss()
        else: 
            val_loss = 0.0
        dt = sum(timei)/len(timei)
        tokens_per_sec = (xs.shape[0] * xs.shape[1] * grad_accum_steps)/dt # (B*T)/dt
        print(f"Step {step}: Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | dt: {dt*1000:.2f}ms | {int(tokens_per_sec):,d} tok/s")

    step += 1

Step 1600: Train Loss: 2.2177 | Val Loss: 0.0000 | dt: 150.96ms | 217,068 tok/s
Step 1700: Train Loss: 2.1816 | Val Loss: 0.0000 | dt: 124.59ms | 263,012 tok/s
Step 1800: Train Loss: 2.0781 | Val Loss: 2.1604 | dt: 122.10ms | 268,368 tok/s
Step 1900: Train Loss: 2.0936 | Val Loss: 0.0000 | dt: 122.17ms | 268,223 tok/s
Step 2000: Train Loss: 2.0665 | Val Loss: 0.0000 | dt: 124.46ms | 263,277 tok/s
Step 2100: Train Loss: 2.0859 | Val Loss: 2.0882 | dt: 121.77ms | 269,100 tok/s
Step 2200: Train Loss: 2.0384 | Val Loss: 0.0000 | dt: 122.76ms | 266,926 tok/s
Step 2300: Train Loss: 2.0144 | Val Loss: 0.0000 | dt: 122.32ms | 267,896 tok/s
Step 2400: Train Loss: 2.0230 | Val Loss: 2.0484 | dt: 129.08ms | 253,864 tok/s
Step 2500: Train Loss: 1.9710 | Val Loss: 0.0000 | dt: 128.87ms | 254,275 tok/s
Step 2600: Train Loss: 1.9778 | Val Loss: 0.0000 | dt: 135.94ms | 241,048 tok/s
Step 2700: Train Loss: 1.9386 | Val Loss: 2.0122 | dt: 124.25ms | 263,721 tok/s
Step 2800: Train Loss: 1.9152 | Val Loss

KeyboardInterrupt: 

In [ ]:
val_loss = get_val_loss()
val_loss

1.836436948776245

In [ ]:
SAVE = False
if SAVE:
    params_list = list(model.parameters())
    torch.save(params_list, "models/params_swiglu.pth")

In [17]:
print(mn.prompt("", model=model, tokenizer=sp, new_tokens=256, debug=True, device=config.device))

</s>
 Lily liked to whistle in the lake. She whistled to all the different animals and the bunns in the peaceful water. She whistled with her bow and whistled. One day, she saw something new in the lake. It was red and green and had hair on it. She swam to it and whistled with it. She whistled to the fish of water and its bark. "Hello, fish!" she said. "Do you want to play?" She took a screen and gave it to the fish. The fish liked the conversation and whistling. Lily whistled aio Gertday. She whistled the same song as Tom whistles. She whistled very well. She whisted and sang a happy boynieve his song. Lilying a song from far away. "Goodould you like to be very good?" she said. She clapped her hands and joined the fish and tickle her with her arms. The fish moved on and made a funny sound. Lily giggled and had fun. She whistled with the fish and whisted for a long time. She felt very happy. When the fish flew away, she waved goodbye to the water and went home. She was so happy and tir